# Embed and Cluster the Topic Phrases

| Item | Description |
|---|---|
| Goal | Group the 9,095 distinct topic phrases of T2.1.1 by meaning. Each phrase is embedded, near-identical phrases are joined into families, and the families are clustered at nine candidate cluster counts with a consensus method that repeats the clustering on bootstrap samples. The result is the set of candidate clusters that T2.1.3 labels and T2.1.4 reviews. |
| Input | `track1/topics.parquet` from T2.1.1 in the T2.1 Interim folder; the encoder BAAI/bge-large-en-v1.5 from the project cache. |
| Output | In `taxonomy/` of the T2.1 Interim folder: `phrases.parquet` and `families.parquet` (phrase to family), `unit_embeddings.npz`, `family_assignment.npz`, `cut_labels.parquet` (the cluster of each family at each candidate count), `cluster_stats.parquet`, and `criteria_curves.csv`. |
| Variables | • `weight`: the number of hearings that mention the phrase; it weights the clustering<br>• `family_id`: the group of near-identical phrases, joined at cosine 0.90 or above<br>• `K150`, `K15`, and the other `K` columns: the cluster of the family at that candidate count<br>• `stability`: the mean share of bootstrap runs in which two members of a cluster fell together; reported, never used as a gate |
| Run | First run 2026-08-22 on one A40; the embeddings, the families, and the consensus results were saved. Run again on 2026-10-31 against those caches: no embedding or clustering is repeated, the phrase and family tables are rebuilt and were found identical. The outputs shown are from that run. |


## Step 1. Import packages


In [1]:
import json
import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score
from transformers import AutoModel, AutoTokenizer


## Step 2. Define file locations


In [2]:
PROJECT_FOLDER = Path.cwd().resolve().parents[1]  # <section>/<subfolder>/ sits two levels below the clone
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis")  # the one line to change on another machine
NOTEBOOK_SUBFOLDER = Path("T2 - Topic Extraction and Stance Analysis/T2.1 - Track 1 - Topic Extraction based on Hearing Summaries")
NOTEBOOK_NAME = "T2.1.2 - Cluster - Embed and Cluster the Topic Phrases"  # the notebook filename without .ipynb
RAW_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Raw"
INTERIM_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Interim"
PROCESSED_FOLDER = SCRATCH_FOLDER / NOTEBOOK_SUBFOLDER / "Processed"
TRACK1_FOLDER = INTERIM_FOLDER / "track1"
TAXONOMY_FOLDER = INTERIM_FOLDER / "taxonomy"
SEED = 1089
FIGURE_FOLDER = PROJECT_FOLDER / "R - Results/figures" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
TABLE_FOLDER = PROJECT_FOLDER / "R - Results/tables" / NOTEBOOK_SUBFOLDER / NOTEBOOK_NAME
MODEL_CACHE = SCRATCH_FOLDER / "cache" / "huggingface"
ENV_FILE = PROJECT_FOLDER / "T2 - Topic Extraction and Stance Analysis" / ".env"
for folder in (INTERIM_FOLDER, PROCESSED_FOLDER):
    folder.mkdir(parents=True, exist_ok=True)
MODEL_CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE)
# A token is needed only for gated models. Qwen3.5-4B and bge-large-en-v1.5 are open; the lines stay so the cell also serves a gated model.
if ENV_FILE.exists():
    for line in ENV_FILE.read_text().splitlines():
        name, _, value = line.strip().partition("=")
        if name == "HF_TOKEN" and value:
            os.environ["HF_TOKEN"] = value.strip().strip('"').strip("'")


## Step 3. Define the encoder call


In [3]:
ENCODER = "BAAI/bge-large-en-v1.5"  # the encoder of T1.3.2 and T3; CLS pooling is the pooling this checkpoint publishes


@torch.no_grad()
def encode(texts, model, tokenizer, batch_size=256, max_length=512):
    """Embed texts: CLS token, L2-normalized, stored in half precision. Cosine similarity is then a dot product."""
    out = np.empty((len(texts), model.config.hidden_size), dtype=np.float16)
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        enc = tokenizer(batch, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(model.device)
        vec = torch.nn.functional.normalize(model(**enc).last_hidden_state[:, 0], dim=-1)
        out[start:start + len(batch)] = vec.float().cpu().numpy().astype(np.float16)
    return out


## Step 4. Build the phrase table

One row per distinct phrase, matched without regard to case. The weight is the number of mentions across hearings.


In [4]:
rng = np.random.default_rng(SEED)
TAXONOMY_FOLDER.mkdir(parents=True, exist_ok=True)

t1_raw = pd.read_parquet(TRACK1_FOLDER / "topics.parquet", columns=["topics_json"])
pool = pd.DataFrame({"phrase": [t["topic"] for js in t1_raw["topics_json"] for t in json.loads(js)]})
pool["key"] = pool["phrase"].str.casefold().str.strip()
units = (pool.groupby("key").agg(phrase=("phrase", lambda s: s.value_counts().index[0]), weight=("phrase", "size")).reset_index())
print(f"Track 1 mentions: {len(pool):,} -> distinct phrases: {len(units):,}")


Track 1 mentions: 11,592 -> distinct phrases: 9,095


## Step 5. Normalize the phrases for clustering

Years, bill numbers, and a trailing "Act" or "reauthorization" are removed for the clustering only, so that "Honest Ads Act" and "Honest Ads" fall together. Phrases that normalize to fewer than three characters go to the explicit other pool.


In [5]:
BILL = re.compile(r"\b(?:h\.?\s?r\.?|s\.?)\s?\d+\b", re.IGNORECASE)
YEAR = re.compile(r"\b(?:19|20)\d{2}\b")
TRAIL = re.compile(r"\s+(?:reauthorization|reauthorizations|act)\s*$", re.IGNORECASE)


def norm_for_clustering(phrase):
    s = BILL.sub(" ", phrase)
    s = YEAR.sub(" ", s)
    s = re.sub(r"\bof\s*$", " ", TRAIL.sub(" ", s))
    s = TRAIL.sub(" ", s)
    return re.sub(r"[\s\-–—,.()]+$", "", re.sub(r"\s+", " ", s)).strip()


units["norm"] = units["phrase"].map(norm_for_clustering)
units["clusterable"] = units["norm"].str.len() >= 3
norm_units = (units[units["clusterable"]].groupby(units["norm"].str.casefold())
              .agg(norm=("norm", "first"), weight=("weight", "sum")).reset_index(names="norm_key"))
print(f"unclusterable -> other: {(~units['clusterable']).sum():,}; clustering units: {len(norm_units):,}")


unclusterable -> other: 79; clustering units: 8,736


## Step 6. Embed the clustering units

The embeddings are cached. When the cache exists it is read back and the encoder is not loaded.


In [6]:
EMB = TAXONOMY_FOLDER / "unit_embeddings.npz"
if EMB.exists():
    dat = np.load(EMB, allow_pickle=True)
    assert list(dat["norm_key"]) == list(norm_units["norm_key"])
    X = dat["X"].astype(np.float32)
    print("reused cached embeddings")
else:
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(ENCODER)
    encoder = AutoModel.from_pretrained(ENCODER, dtype="auto", device_map=device).eval()
    X = encode(norm_units["norm"].tolist(), encoder, tokenizer).astype(np.float32)
    np.savez_compressed(EMB, X=X.astype(np.float16), norm_key=np.array(norm_units["norm_key"]))
    del encoder
    torch.cuda.empty_cache()
print(f"X {X.shape}")


reused cached embeddings
X (8736, 1024)


## Step 7. Join near-identical phrases into families

Average-linkage clustering at cosine 0.90 or above. Single linkage was tried earlier and chained thousands of units into one family; average linkage does not.


In [7]:
FAMILY_COS = 0.90
FAM_CACHE = TAXONOMY_FOLDER / "family_assignment.npz"
if FAM_CACHE.exists():
    cached = np.load(FAM_CACHE)
    assert len(cached["fam"]) == len(norm_units)
    fam = cached["fam"]
    print("reused cached family assignment")
else:
    Xg = torch.tensor(X, device="cuda" if torch.cuda.is_available() else "cpu")
    Du = np.empty((len(X), len(X)), dtype=np.float32)
    for i in range(0, len(X), 4096):
        Du[i:i + 4096] = (1.0 - (Xg[i:i + 4096] @ Xg.T)).clamp(min=0).cpu().numpy()
    del Xg
    torch.cuda.empty_cache()
    np.fill_diagonal(Du, 0.0)
    Zu = linkage(squareform(Du, checks=False), method="average")
    del Du
    fam = fcluster(Zu, t=1.0 - FAMILY_COS, criterion="distance") - 1
    np.savez_compressed(FAM_CACHE, fam=fam, threshold=FAMILY_COS)
n_fam = fam.max() + 1
norm_units["family_id"] = fam

w = norm_units["weight"].to_numpy(dtype=np.float64)
F = np.zeros((n_fam, X.shape[1]), dtype=np.float64)
np.add.at(F, fam, X * w[:, None])
F /= np.linalg.norm(F, axis=1, keepdims=True)
F = F.astype(np.float32)
fam_w = np.bincount(fam, weights=w)
sizes = np.bincount(fam)
print(f"{n_fam:,} families (max size {sizes.max()}, singletons {(sizes == 1).sum():,})")
assert sizes.max() < 500


reused cached family assignment


7,712 families (max size 15, singletons 6,937)


## Step 8. Cluster the families at nine candidate counts

For each candidate count K, the families are clustered 30 times on weight-proportional 80 percent samples. The share of runs in which two families fall together is their co-assignment. Average linkage on the co-assignment matrix, cut at K, gives the clusters. Three criteria are recorded per K. They are the weighted mean co-assignment within clusters (stability), a sampled silhouette, and the count of clusters with scattered members (cohesion flags). All three are flat across the candidate counts, so they did not pick a number. The topic level was set at 150 clusters so that the candidates sit at a granularity comparable with CAP's subtopics, the external yardstick of T2.3. The review in T2.1.4 then reduced them to 111. Stability is reported per cluster in the codebook and gates nothing.


In [8]:
B = 30
GRID = [15, 20, 25, 40, 60, 80, 100, 120, 150]
DONE = all((TAXONOMY_FOLDER / f).exists() for f in ("criteria_curves.csv", "cut_labels.parquet", "cluster_stats.parquet"))
if DONE:
    curves = pd.read_csv(TAXONOMY_FOLDER / "criteria_curves.csv")
    DONE = list(curves["K"]) == GRID
    if DONE:
        print("reused cached per-K consensus results")
        print(curves.to_string(index=False))
if not DONE:
    p = fam_w / fam_w.sum()
    sil_idx = rng.choice(n_fam, size=min(4000, n_fam), replace=False, p=p)
    curves, cut_labels, cluster_stats = [], {}, []
    for K in GRID:
        co = np.zeros((n_fam, n_fam), dtype=np.uint8)
        for b in range(B):
            sub = rng.choice(n_fam, size=int(0.8 * n_fam), replace=False, p=p)
            km = MiniBatchKMeans(n_clusters=K, n_init=3, batch_size=4096, random_state=int(rng.integers(2**31)))
            km.fit(F[sub], sample_weight=fam_w[sub])
            labels = km.predict(F)
            for k in range(K):
                idx = np.flatnonzero(labels == k)
                co[np.ix_(idx, idx)] += 1
        D = 1.0 - co.astype(np.float32) / B
        np.fill_diagonal(D, 0.0)
        lab = fcluster(linkage(squareform(D, checks=False), method="average"), t=K, criterion="maxclust")
        del D
        cut_labels[K] = lab
        stab = np.zeros(K); coh = np.zeros(K); wsum = np.zeros(K)
        for k in range(1, K + 1):
            idx = np.flatnonzero(lab == k)
            wsum[k - 1] = fam_w[idx].sum()
            stab[k - 1] = co[np.ix_(idx, idx)].mean() / B if len(idx) > 1 else 1.0
            coh[k - 1] = float(np.linalg.norm(F[idx].mean(0)) ** 2)
            cluster_stats.append({"K": K, "cluster": k, "weight": wsum[k - 1], "n_fam": len(idx), "stability": stab[k - 1], "cohesion": coh[k - 1]})
        del co
        curves.append({"K": K, "stability_w": (stab * wsum).sum() / wsum.sum(),
                       "silhouette": silhouette_score(F[sil_idx], lab[sil_idx], metric="cosine"),
                       "cohesion_flags": int((coh < 0.55).sum()), "share_stab50": wsum[stab >= 0.5].sum() / fam_w.sum()})
        print(f"  K={K:>3}: stability {curves[-1]['stability_w']:.2f}, silhouette {curves[-1]['silhouette']:.3f}, cohesion flags {curves[-1]['cohesion_flags']}")
    curves = pd.DataFrame(curves)
    pd.DataFrame({f"K{K}": v for K, v in cut_labels.items()}).assign(family_id=np.arange(n_fam)).to_parquet(TAXONOMY_FOLDER / "cut_labels.parquet", index=False)
    pd.DataFrame(cluster_stats).to_parquet(TAXONOMY_FOLDER / "cluster_stats.parquet", index=False)
    curves.to_csv(TAXONOMY_FOLDER / "criteria_curves.csv", index=False)
    print(curves.to_string(index=False))


reused cached per-K consensus results
  K  stability_w  silhouette  cohesion_flags  share_stab50
 15     0.524940    0.047182               5      0.430545
 20     0.483788    0.042386               2      0.350447
 25     0.471012    0.007551               5      0.353227
 40     0.463762    0.026935               5      0.378594
 60     0.443439    0.014966               4      0.289375
 80     0.459725    0.036738               4      0.357137
100     0.465815    0.023837               4      0.319520
120     0.463900    0.027363               6      0.351316
150     0.474066    0.035660               3      0.389975


## Step 9. Save the phrase-to-family tables


In [9]:
cl = units[units["clusterable"]].copy()
nk = {n: i for i, n in enumerate(norm_units["norm_key"])}
cl["unit_id"] = cl["norm"].str.casefold().map(nk)
cl["family_id"] = norm_units["family_id"].to_numpy()[cl["unit_id"]]
other = units[~units["clusterable"]].assign(unit_id=-1, family_id=-1)
phrases = pd.concat([cl, other], ignore_index=True)
phrases.to_parquet(TAXONOMY_FOLDER / "phrases.parquet", index=False)
fam_tbl = (cl.groupby("family_id")
           .agg(representative=("phrase", lambda s: s.iloc[np.argmax(cl.loc[s.index, "weight"].values)]),
                weight=("weight", "sum"), n_phrases=("phrase", "size")).reset_index())
fam_tbl.to_parquet(TAXONOMY_FOLDER / "families.parquet", index=False)
print(f"phrases {len(phrases):,} | families {len(fam_tbl):,}")


phrases 9,095 | families 7,712


## Step 10. Check the run


In [10]:
back = pd.read_parquet(TAXONOMY_FOLDER / "phrases.parquet")
assert back["weight"].sum() == units["weight"].sum()
print(f"1) Track 1 phrases : {len(units):,} distinct / {units['weight'].sum():,} mentions")
print(f"2) families        : {n_fam:,} (average linkage, largest {sizes.max()})")
print(f"3) consensus       : {B} replicates per K, grid {GRID}")
print(f"4) stability       : reported per cluster, no gate")


1) Track 1 phrases : 9,095 distinct / 11,592 mentions
2) families        : 7,712 (average linkage, largest 15)
3) consensus       : 30 replicates per K, grid [15, 20, 25, 40, 60, 80, 100, 120, 150]
4) stability       : reported per cluster, no gate
